# AI-Guard: Baseline AI-Generated Text Detector Training
### Research Baseline: Logistic Regression + TF-IDF on the Human-ChatGPT Comparison Corpus (HC3)

**Project:** AI-Guard: A Robust AI-Generated Text Detection and Analysis System  
**Target Environment:** Google Colab (Standard CPU or T4 GPU runtime)  
**Author:** AI-Guard Research Team  

---

## Notebook Overview & Purpose
This notebook trains and evaluates an empirical baseline classifier for distinguishing human-written text from AI-generated (ChatGPT) text using the authentic **HC3 (Human ChatGPT Comparison Corpus)** published by *Hello-SimpleAI*.

### Key Highlights:
1. **Environment Compatibility:** Resolves the Colab `RuntimeError: Dataset scripts are no longer supported, but found HC3.py` and `huggingface-hub` dependency collisions.
2. **Official HC3 Data Source:** Loads the verified `all.jsonl` directly via the Hugging Face `json` builder, bypassing legacy loading scripts with zero token requirement.
3. **Zero-Leakage Grouped Splitting:** Groups samples by `question_id` to strictly prevent topic leakage between Train, Validation, and Test sets.
4. **Strict Methodological Discipline:** Hyperparameter selection is conducted on the **Validation set only**. The **Test set** is evaluated once on the final pipeline.
5. **Artifact Export & FastAPI Integration:** Exports the production pipeline, metrics, confusion matrix, metadata, and test predictions, ready for local FastAPI serving.


---
## 1. Environment Setup & Dependency Conflict Resolution

### Diagnosis of Known Colab Issues:
- **Issue 1 (`RuntimeError: Dataset scripts are no longer supported, but found HC3.py`):**
  In `datasets >= 3.0.0`, Hugging Face deprecated and disabled arbitrary Python dataset loading scripts (`*.py`) for security and Hub standardization. Calling `load_dataset('Hello-SimpleAI/HC3')` discovers `HC3.py` and raises a fatal `RuntimeError`.
- **Issue 2 (`huggingface-hub` Dependency Warnings):**
  Pre-installed Google Colab packages (`gradio-client`, `diffusers`, `tokenizers`) pin constraints such as `huggingface-hub < 2.0` (or `< 1.0`). Running unconstrained `pip install -U` triggers dependency conflicts.

### Solution:
We install compatible versions and bypass the legacy script by loading the raw `.jsonl` data directly through Hugging Face's built-in `json` dataset loader.


In [ ]:
# Cell 1: Install required libraries safely without breaking Colab pre-installed packages
!pip install --quiet "datasets>=2.19.0" "huggingface-hub>=0.20.0" scikit-learn joblib pandas numpy matplotlib seaborn

print("Installation complete. Proceed to version verification.")


In [ ]:
# Cell 2: Verify environment and package versions
import sys
import datasets
import huggingface_hub
import pyarrow
import sklearn
import pandas as pd
import numpy as np

print("=" * 60)
print("AI-GUARD ENVIRONMENT VERIFICATION")
print("=" * 60)
print(f"Python Version:          {sys.version.split()[0]}")
print(f"datasets Version:        {datasets.__version__}")
print(f"huggingface-hub Version: {huggingface_hub.__version__}")
print(f"pyarrow Version:         {pyarrow.__version__}")
print(f"scikit-learn Version:    {sklearn.__version__}")
print(f"pandas Version:          {pd.__version__}")
print(f"numpy Version:           {np.__version__}")
print("=" * 60)
print("Status: Environment is compatible and ready.")
print("Note: If you upgraded packages and see binary mismatch warnings, select:")
print("      Runtime -> Restart session, then run from this cell onward.")


---
## 2. Resolving HC3 Dataset Loading

### Dataset Architecture:
The `Hello-SimpleAI/HC3` repository contains:
- `all.jsonl` (70.31 MB) — Complete unified corpus across QA, Reddit, Wikipedia, Finance, and Medicine.
- Subset files: `open_qa.jsonl` (2.78 MB), `wiki_csai.jsonl` (2.10 MB), `medicine.jsonl` (2.56 MB), `finance.jsonl` (9.43 MB), `reddit_eli5.jsonl` (52.88 MB).
- `HC3.py` — The legacy script that is now deprecated by Hugging Face.

We use `hf_hub_download` to fetch the verified `all.jsonl` directly, then load it via `load_dataset('json', data_files=...)`. This is **100% reliable**, does not trigger `HC3.py`, and requires **no authentication token**.


In [ ]:
# Cell 3: Download verified all.jsonl from Hugging Face Hub and load via json builder
import os
from datasets import load_dataset
from huggingface_hub import hf_hub_download

REPO_ID = "Hello-SimpleAI/HC3"
DATASET_FILENAME = "all.jsonl"

print(f"Fetching verified {DATASET_FILENAME} from {REPO_ID}...")
local_data_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=DATASET_FILENAME,
    repo_type="dataset"
)

print(f"Successfully downloaded to: {local_data_path}")
print(f"File size: {os.path.getsize(local_data_path) / (1024 * 1024):.2f} MB")

# Load using Hugging Face datasets native json builder
raw_dataset = load_dataset("json", data_files=local_data_path, split="train")
print(f"\nLoaded raw HC3 dataset: {len(raw_dataset):,} question records.")


In [ ]:
# Cell 4: Inspect schema and example record
print("Dataset Columns:", raw_dataset.column_names)
sample = raw_dataset[0]

print("\nExample Record Inspection:")
print("=" * 60)
print(f"Question: {sample.get('question', '')[:100]}...")
print(f"Source:   {sample.get('source', 'N/A')}")
print(f"Human Answers Count:   {len(sample.get('human_answers', []))}")
print(f"ChatGPT Answers Count: {len(sample.get('chatgpt_answers', []))}")

if sample.get('human_answers'):
    print(f"\n[Human Answer Sample]:\n{sample['human_answers'][0][:150]}...")
if sample.get('chatgpt_answers'):
    print(f"\n[ChatGPT Answer Sample]:\n{sample['chatgpt_answers'][0][:150]}...")


---
## 3. Dataset Preprocessing, Normalization & Grouped Splitting

### Crucial Methodological Safeguards:
1. **Unrolling Answers:**
   Each question contains lists of human answers and ChatGPT answers. We unroll these into individual `(text, label, question_id)` rows.
   - `Label 0`: Human-written text
   - `Label 1`: AI-generated text (ChatGPT)
2. **Text Normalization & Deduplication:**
   Normalize excess whitespace, drop trivial snippets (< 20 characters), remove identical duplicate texts, and detect/eliminate contradictory labels.
3. **Zero-Leakage Grouped Splitting:**
   Standard random splitting causes catastrophic **topic leakage**: if question $Q_i$ appears in both Train and Test, the classifier memorizes topic keywords (e.g., specific historical figures or scientific terms) instead of authentic stylistic signatures.
   We apply **`GroupShuffleSplit` on `question_id`** so all answers to any given question remain strictly inside one split!


In [ ]:
# Cell 5: Unroll answers, clean whitespace, and deduplicate
import re
import pandas as pd

def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # Standardize whitespace and remove non-printable characters
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

records = []
for q_id, row in enumerate(raw_dataset):
    q_text = normalize_text(row.get("question", ""))
    src = row.get("source", "unknown")
    
    # Label 0: Human
    for h_ans in row.get("human_answers", []):
        cleaned = normalize_text(h_ans)
        if len(cleaned) >= 25:  # Require meaningful passage length
            records.append({
                "question_id": q_id,
                "question": q_text,
                "text": cleaned,
                "label": 0,
                "source": src
            })
            
    # Label 1: AI (ChatGPT)
    for a_ans in row.get("chatgpt_answers", []):
        cleaned = normalize_text(a_ans)
        if len(cleaned) >= 25:
            records.append({
                "question_id": q_id,
                "question": q_text,
                "text": cleaned,
                "label": 1,
                "source": src
            })

df = pd.DataFrame(records)
print(f"Total unrolled raw samples: {len(df):,}")

# Check for conflicting duplicate labels (same text with both label 0 and 1)
label_counts_per_text = df.groupby("text")["label"].nunique()
conflicting_texts = label_counts_per_text[label_counts_per_text > 1].index
print(f"Conflicting text samples found: {len(conflicting_texts):,}")
if len(conflicting_texts) > 0:
    df = df[~df["text"].isin(conflicting_texts)]

# Drop exact duplicate texts
initial_len = len(df)
df = df.drop_duplicates(subset=["text"], keep="first").reset_index(drop=True)
print(f"Removed {initial_len - len(df):,} duplicate texts.")
print(f"Cleaned unique dataset size: {len(df):,} samples across {df['question_id'].nunique():,} questions.")

print("\nClass Counts:")
print(df["label"].value_counts().rename({0: "0 (Human-Written)", 1: "1 (AI-Generated)"}))


In [ ]:
# Cell 6: Perform leakage-free grouped splitting (Train 70%, Validation 15%, Test 15%)
from sklearn.model_selection import GroupShuffleSplit

# Outer split: Train (70%) vs Temp (30%)
gss_outer = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=42)
train_idx, temp_idx = next(gss_outer.split(df, groups=df["question_id"]))

df_train = df.iloc[train_idx].copy().reset_index(drop=True)
df_temp = df.iloc[temp_idx].copy().reset_index(drop=True)

# Inner split: Temp split 50/50 -> Validation (15% total), Test (15% total)
gss_inner = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=42)
val_idx, test_idx = next(gss_inner.split(df_temp, groups=df_temp["question_id"]))

df_val = df_temp.iloc[val_idx].copy().reset_index(drop=True)
df_test = df_temp.iloc[test_idx].copy().reset_index(drop=True)

# Strict assertion that no question crosses split boundaries
train_q = set(df_train["question_id"])
val_q = set(df_val["question_id"])
test_q = set(df_test["question_id"])

assert len(train_q.intersection(val_q)) == 0, "LEAKAGE ERROR: Overlapping questions between Train and Val!"
assert len(train_q.intersection(test_q)) == 0, "LEAKAGE ERROR: Overlapping questions between Train and Test!"
assert len(val_q.intersection(test_q)) == 0, "LEAKAGE ERROR: Overlapping questions between Val and Test!"

print("=" * 60)
print("GROUPED DATASET SPLIT SUMMARY (ZERO QUESTION OVERLAP)")
print("=" * 60)
print(f"Train Set:      {len(df_train):,} samples ({len(train_q):,} questions) - {len(df_train)/len(df)*100:.1f}%")
print(f"Validation Set: {len(df_val):,} samples ({len(val_q):,} questions) - {len(df_val)/len(df)*100:.1f}%")
print(f"Test Set:       {len(df_test):,} samples ({len(test_q):,} questions) - {len(df_test)/len(df)*100:.1f}%")
print("=" * 60)
print("Train class ratio: ", df_train["label"].value_counts(normalize=True).round(3).to_dict())
print("Val class ratio:   ", df_val["label"].value_counts(normalize=True).round(3).to_dict())
print("Test class ratio:  ", df_test["label"].value_counts(normalize=True).round(3).to_dict())


---
## 4. Train the Baseline Model Pipeline

### Pipeline Components:
1. **`TfidfVectorizer`:** Extracts unigrams and bigrams with sublinear term-frequency scaling (`sublinear_tf=True`), pruning rare noise tokens (`min_df=3`) and normalizing unicode characters.
2. **`LogisticRegression`:** L2-regularized linear classifier with balanced class weighting and reproducible random state.

### Hyperparameter Optimization Discipline:
To prevent data leakage, hyperparameter selection is strictly evaluated against the **Validation set**. The **Test set** is completely held out until final reporting.


In [ ]:
# Cell 7: Hyperparameter exploration on Validation set
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

X_train, y_train = df_train["text"], df_train["label"]
X_val, y_val = df_val["text"], df_val["label"]
X_test, y_test = df_test["text"], df_test["label"]

candidate_configs = [
    {"name": "Config A (Unigrams, C=1.0)", "ngram_range": (1, 1), "max_features": 20000, "C": 1.0},
    {"name": "Config B (Uni+Bigrams, C=1.0)", "ngram_range": (1, 2), "max_features": 35000, "C": 1.0},
    {"name": "Config C (Uni+Bigrams, C=2.5)", "ngram_range": (1, 2), "max_features": 50000, "C": 2.5},
    {"name": "Config D (Uni+Bigrams, C=5.0)", "ngram_range": (1, 2), "max_features": 50000, "C": 5.0},
]

print("Evaluating candidate configurations on VALIDATION set...")
print("=" * 65)

best_val_f1 = -1.0
best_pipeline = None
best_config = None

for cfg in candidate_configs:
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=cfg["ngram_range"],
            max_features=cfg["max_features"],
            sublinear_tf=True,
            min_df=3,
            strip_accents="unicode"
        )),
        ("clf", LogisticRegression(
            C=cfg["C"],
            max_iter=1000,
            class_weight="balanced",
            solver="lbfgs",
            random_state=42
        ))
    ])
    
    pipe.fit(X_train, y_train)
    val_preds = pipe.predict(X_val)
    acc = accuracy_score(y_val, val_preds)
    f1 = f1_score(y_val, val_preds, pos_label=1)
    
    print(f"{cfg['name']:<35} | Val Acc: {acc*100:.2f}% | Val F1: {f1*100:.2f}%")
    
    if f1 > best_val_f1:
        best_val_f1 = f1
        best_pipeline = pipe
        best_config = cfg

print("=" * 65)
print(f"Selected Optimal Configuration: {best_config['name']}")
print(f"Validation F1-Score: {best_val_f1*100:.2f}%")


---
## 5. Model Evaluation on Held-Out Test Set

We evaluate the selected pipeline on the unseen **Test set** and calculate:
- **Accuracy, Precision (AI), Recall (AI), F1-Score (AI), Macro-F1**
- **ROC-AUC**
- **Confusion Matrix** (True Human, False AI, Missed AI, Detected AI)
- Full **Classification Report**

All metrics are exported to `metrics.json` and plotted to `confusion_matrix.png`.


In [ ]:
# Cell 8: Compute test set evaluation metrics and generate confusion matrix plot
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# Generate test predictions
test_preds = best_pipeline.predict(X_test)
test_probs = best_pipeline.predict_proba(X_test)[:, 1]

# Compute metrics
acc = float(accuracy_score(y_test, test_preds))
prec = float(precision_score(y_test, test_preds, pos_label=1))
rec = float(recall_score(y_test, test_preds, pos_label=1))
f1 = float(f1_score(y_test, test_preds, pos_label=1))
macro_f1 = float(f1_score(y_test, test_preds, average="macro"))
roc_auc = float(roc_auc_score(y_test, test_probs))
cm = confusion_matrix(y_test, test_preds).tolist()
report_dict = classification_report(y_test, test_preds, target_names=["Human (0)", "AI (1)"], output_dict=True)

metrics_data = {
    "accuracy": round(acc, 4),
    "precision_ai": round(prec, 4),
    "recall_ai": round(rec, 4),
    "f1_score_ai": round(f1, 4),
    "macro_f1": round(macro_f1, 4),
    "roc_auc": round(roc_auc, 4),
    "confusion_matrix": {
        "true_negative_human": cm[0][0],
        "false_positive_human_flagged_as_ai": cm[0][1],
        "false_negative_ai_missed": cm[1][0],
        "true_positive_ai_detected": cm[1][1]
    },
    "test_sample_size": len(y_test),
    "selected_hyperparameters": best_config
}

print("=" * 60)
print("HELD-OUT TEST SET EVALUATION RESULTS")
print("=" * 60)
print(f"Accuracy:          {acc * 100:.2f}%")
print(f"Precision (AI):    {prec * 100:.2f}%")
print(f"Recall (AI):       {rec * 100:.2f}%")
print(f"F1-Score (AI):     {f1 * 100:.2f}%")
print(f"Macro F1:          {macro_f1 * 100:.2f}%")
print(f"ROC-AUC:           {roc_auc:.4f}")
print("=" * 60)
print("\nClassification Report:")
print(classification_report(y_test, test_preds, target_names=["Human (0)", "AI (1)"]))

# Save metrics to JSON
with open("metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_data, f, indent=2)
print("Saved test metrics to metrics.json")

# Plot Confusion Matrix
plt.figure(figsize=(6.5, 5), dpi=300)
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Pred: Human (0)", "Pred: AI (1)"],
    yticklabels=["True: Human (0)", "True: AI (1)"],
    cbar=False,
    annot_kws={"size": 13, "weight": "bold"}
)
plt.title(f"AI-Guard Baseline: Test Confusion Matrix\n(Accuracy: {acc*100:.1f}%, F1: {f1*100:.1f}%)", fontsize=12, pad=12)
plt.ylabel("Ground Truth", fontsize=11)
plt.xlabel("Model Prediction", fontsize=11)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300)
plt.show()
print("Saved plot to confusion_matrix.png")


---
## 6. Model Export, Predictions CSV & Colab Download

We save the required artifacts:
1. `ai_guard_baseline.joblib` — The trained scikit-learn pipeline (TF-IDF + Logistic Regression).
2. `metrics.json` — Evaluated test metrics and parameters.
3. `metadata.json` — Model specification, dataset splits, framework versions, and research disclaimer.
4. `confusion_matrix.png` — Publication-ready confusion matrix visualization.
5. `test_predictions.csv` — Full test inference log with true labels, predicted labels, and probabilities.

Includes a **smoke test cell** reloading the `.joblib` model and verifying predictions.


In [ ]:
# Cell 9: Save artifacts, export test predictions CSV, and run smoke test
import joblib
from datetime import datetime

# 1. Save trained model pipeline
MODEL_FILE = "ai_guard_baseline.joblib"
joblib.dump(best_pipeline, MODEL_FILE)
print(f"Saved pipeline model to: {MODEL_FILE}")

# 2. Save metadata.json
metadata = {
    "project": "AI-Guard: Robust AI-Generated Text Detection",
    "model_name": "AI-Guard TF-IDF Logistic Regression Baseline",
    "version": "1.0.0",
    "created_at": datetime.utcnow().isoformat() + "Z",
    "framework": "scikit-learn",
    "sklearn_version": sklearn.__version__,
    "dataset": {
        "name": "Hello-SimpleAI/HC3",
        "file": "all.jsonl",
        "train_samples": len(df_train),
        "val_samples": len(df_val),
        "test_samples": len(df_test),
        "split_strategy": "GroupShuffleSplit on question_id (zero question leakage)"
    },
    "pipeline_parameters": {
        "ngram_range": list(best_config["ngram_range"]),
        "max_features": best_config["max_features"],
        "C": best_config["C"],
        "class_weight": "balanced"
    },
    "metrics": {
        "accuracy": acc,
        "f1_score": f1,
        "roc_auc": roc_auc
    },
    "disclaimer": "Probabilistic estimate for academic evaluation only. Not proof of authorship."
}

with open("metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)
print("Saved metadata.json")

# 3. Save test_predictions.csv
df_predictions = pd.DataFrame({
    "question_id": df_test["question_id"],
    "source": df_test.get("source", "unknown"),
    "true_label": y_test,
    "predicted_label": test_preds,
    "ai_probability": np.round(test_probs, 4),
    "text_snippet": df_test["text"].apply(lambda t: t[:200] + ("..." if len(t) > 200 else ""))
})
df_predictions.to_csv("test_predictions.csv", index=False)
print(f"Saved test_predictions.csv ({len(df_predictions):,} rows)")

# 4. Smoke Test: Reload pipeline and test inference
print("\n" + "=" * 60)
print("RELOAD SMOKE TEST: VERIFYING PREDICTION ON NEW SAMPLES")
print("=" * 60)
reloaded_model = joblib.load(MODEL_FILE)

test_prompts = [
    "The humanities include ancient and modern languages, literature, philosophy, religion, and visual and performing arts.",
    "Artificial intelligence has fundamentally transformed the landscape of modern education by providing personalized tutoring systems.",
    "In my hometown, the municipal bus system is frequently delayed during heavy morning rainfall."
]

preds = reloaded_model.predict(test_prompts)
probs = reloaded_model.predict_proba(test_prompts)[:, 1]

for prompt, p, prob in zip(test_prompts, preds, probs):
    label_str = "AI-Generated (1)" if p == 1 else "Human-Written (0)"
    print(f"Text: '{prompt[:70]}...'")
    print(f"  -> Result: {label_str} | AI Likelihood: {prob*100:.1f}%\n")

print("Smoke test passed! Model is ready for FastAPI deployment.")


In [ ]:
# Cell 10: Bundle and download all artifacts in Google Colab
import zipfile

zip_name = "ai_guard_baseline_artifacts.zip"
files_to_pack = [
    "ai_guard_baseline.joblib",
    "metrics.json",
    "metadata.json",
    "confusion_matrix.png",
    "test_predictions.csv"
]

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in files_to_pack:
        if os.path.exists(fname):
            zf.write(fname)
            print(f"Archived: {fname}")

print(f"\nCreated archive: {zip_name} ({os.path.getsize(zip_name)/(1024*1024):.2f} MB)")

# Trigger browser download in Colab
try:
    from google.colab import files
    print("Triggering Google Colab browser download...")
    files.download(zip_name)
except ImportError:
    print("Running outside Google Colab. Artifacts are stored in the current working directory.")


---
## 7. FastAPI Backend Integration Guide

Here is the complete FastAPI endpoint implementation to serve the exported `ai_guard_baseline.joblib` model.

### Directory Placement:
Place `ai_guard_baseline.joblib` inside your backend repository:
```
backend/
├── app/
│   ├── main.py
│   └── models/
│       └── ai_guard_baseline.joblib
└── requirements.txt
```

### Implementation (`main.py`):
```python
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from contextlib import asynccontextmanager
import joblib
import os

MODEL_PATH = os.getenv("MODEL_PATH", "models/ai_guard_baseline.joblib")
ml_models = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Load model on startup
    if not os.path.exists(MODEL_PATH):
        raise RuntimeError(f"Model artifact not found at {MODEL_PATH}")
    ml_models["baseline"] = joblib.load(MODEL_PATH)
    yield
    ml_models.clear()

app = FastAPI(title="AI-Guard Inference API", version="1.0.0", lifespan=lifespan)

class AnalyzeRequest(BaseModel):
    text: str = Field(..., min_length=20, description="Text passage to evaluate (min 20 chars)")

class AnalyzeResponse(BaseModel):
    prediction: str
    label: int
    ai_probability: float
    human_probability: float
    confidence_score: float
    word_count: int
    disclaimer: str

@app.post("/api/detect", response_model=AnalyzeResponse)
async def detect_ai_text(req: AnalyzeRequest):
    text = req.text.strip()
    if len(text.split()) < 5:
        raise HTTPException(status_code=400, detail="Text must contain at least 5 words for analysis.")
    
    model = ml_models.get("baseline")
    if model is None:
        raise HTTPException(status_code=503, detail="Classifier model not initialized.")
    
    # Predict class and probabilities
    pred_label = int(model.predict([text])[0])
    probs = model.predict_proba([text])[0]
    prob_human = float(probs[0])
    prob_ai = float(probs[1])
    
    # Confidence is distance from decision boundary (0.50)
    confidence = round(abs(prob_ai - 0.50) * 2.0, 4)
    
    return AnalyzeResponse(
        prediction="AI-Generated" if pred_label == 1 else "Human-Written",
        label=pred_label,
        ai_probability=round(prob_ai, 4),
        human_probability=round(prob_human, 4),
        confidence_score=confidence,
        word_count=len(text.split()),
        disclaimer="Probabilistic estimate for academic research only. Does not prove authorship."
    )
```


---
## 8. Research Limitations, Bias & Academic Disclaimer

### 1. Dataset Limitations (HC3):
- **Question-Answering Domain Concentration:** HC3 focuses primarily on informational Q&A (Reddit ELI5, Wikipedia, open questions). Performance may degrade on creative narrative fiction, technical source code, or conversational dialogue.
- **Model Generator Representation:** HC3 features ChatGPT responses collected in late 2022 / early 2023. Frontier models (e.g., GPT-4o, Claude 3.5, Gemini 1.5) exhibit nuanced stylistic shifts not fully captured by early models.

### 2. Topic & Style Confounding:
- Despite our strict **question-grouped splitting**, linear n-gram models can still pick up stylistic mannerisms (e.g., "Furthermore", "In conclusion") rather than deep semantic discourse cohesion.

### 3. False Positives & Non-Native (ESL) Bias:
- Non-native English writers often utilize standardized, formulaic sentence structures and restricted lexicons, which shallow statistical detectors frequently misclassify as AI-generated. **This detector must never be used punitively or as sole evidence of academic misconduct.**

### 4. Adversarial Fragility:
- Light paraphrasing, synonym substitution (e.g., QuillBot), or homoglyphic unicode perturbations can significantly degrade TF-IDF features. See the AI-Guard frontend Robustness Lab for quantitative evasion benchmarks.

---
**Citation & Acknowledgement:**
```bibtex
@article{guo2023hc3,
  title={How Close is ChatGPT to Human Experts? Comparison Corpus, Evaluation, and Detection},
  author={Guo, Biyang and Zhang, Xin and Wang, Ziyuan and Jiang, Minlie and Nie, Jinqiu and Ding, Yuxian and Yue, Jianwei and Wu, Yupeng},
  journal={arXiv preprint arXiv:2301.07597},
  year={2023}
}
```
